# 00 · Download the complete calcium-imaging and physiology dataset

Run this notebook once before the analysis notebooks. It downloads every
version-3 recording used by notebooks 01–09:

- 10 processed calcium-imaging files (about 11.22 GB total); and
- synchronized EEG/EMG files plus their README (about 4.06 GB total).

The complete download is about **15.3 GB**. There is intentionally no
single-recording mode: the repository now uses one consistent local dataset
for both the worked examples and the cohort analyses.

Dataset version 3: RIKEN neurodata 20260708-001 (CC-BY 4.0)
https://neurodata.riken.jp/id/20260708-001

Files are streamed from the public RIKEN API. A file that already has the
expected byte size is skipped, so rerunning this notebook is safe after an
interrupted download. A partial file is never renamed to its final name until
its size has been validated.

In [ ]:
# ruff: noqa: E402
from __future__ import annotations

import sys
from pathlib import Path

import requests
from tqdm.auto import tqdm

# Notebook kernels may start in the repository root, ``scripts/``, or another
# subdirectory. Search upward for the folder that contains ``src/funcnet``.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "funcnet").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Open this notebook from inside the cloned ca_imaging_network_analysis "
        "repository so its src/funcnet package can be found."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.funcnet.paths import PHYSIOLOGY_DIR, RAW_DIR

API = "https://neurodata.riken.jp/api/v3/files/{id}/download/"

CALCIUM_FILES: dict[str, tuple[str, int]] = {
    "mouse01_sleep.mat": ("1f786027-c4b2-4ac9-8449-24c0cfb434db", 658_474_258),
    "mouse02_sleep.mat": ("b266f5ee-cc1d-4a47-9d41-29655a7c3967", 1_093_789_677),
    "mouse03_sleep.mat": ("3191b169-1d10-4807-9284-edfd96f6c44e", 1_197_235_306),
    "mouse03_ane.mat": ("1616a6b6-edec-4637-bea9-bca9c0a95848", 1_123_058_079),
    "mouse04_day1_sleep.mat": ("8d84d7ac-b4c8-4208-9e5c-c3a4605d3e31", 1_625_754_526),
    "mouse04_day2_sleep.mat": ("105e2c10-8a9a-4cd2-9c76-411cf673ec28", 2_254_368_168),
    "mouse05_sleep.mat": ("ed06e6f2-7485-493e-8d32-d01e0d13aec9", 1_263_665_343),
    "mouse05_ane.mat": ("5b5f67db-2bd0-43ae-8b4f-eb50138e6057", 1_079_340_853),
    "mouse06_ane.mat": ("e1dd0560-3486-45e6-a086-e2c1bfb2bb6e", 563_929_318),
    "mouse07_ane.mat": ("ffdc68e6-fbb4-43f5-81e2-a8271a50819a", 363_189_208),
}

PHYSIOLOGY_FILES: dict[str, tuple[str, int]] = {
    "README_EEG_EMG.md": ("62efc349-d2ac-4aec-89d5-d747af61a20c", 2_969),
    "mouse01_sleep_physiological_data.mat": ("ba79f124-a00e-4f71-ae60-729ee9b8dbcc", 180_001_112),
    "mouse02_sleep_physiological_data.mat": ("b60cfc5d-c6f5-499f-bb8b-e93564e07360", 360_001_112),
    "mouse03_sleep_physiological_data.mat": ("1b58e659-c2d9-406d-94dd-41d25c856f78", 360_001_112),
    "mouse04_day1_sleep_physiological_data.mat": ("6cf82a55-0e08-4a07-8716-9aed78b50430", 360_001_112),
    "mouse04_day2_sleep_physiological_data.mat": ("25761808-4bbd-471f-b249-4f48d0cb33d4", 432_001_112),
    "mouse05_sleep_physiological_data.mat": ("3c6a3b7b-3dfb-4631-a670-5edbf8ebf81c", 360_001_112),
    "mouse03_ane_physiological_data.mat": ("0bf7785e-612a-41cd-9f52-33261659be30", 362_601_312),
    "mouse05_ane_physiological_data.mat": ("48f04ef2-dcd8-48fc-8d16-26f079cfb314", 546_001_312),
    "mouse06_ane_physiological_data_awake.mat": ("e80affb3-91a2-4a7c-bafc-5cfe352b3fca", 179_201_296),
    "mouse06_ane_physiological_data_ane.mat": ("b5548e9c-76a4-4b7a-8a11-b58fc48a332f", 299_601_296),
    "mouse07_ane_physiological_data.mat": ("4a57ad37-a7e9-4948-acc9-708926eb1b4f", 619_201_496),
}

## Download helpers

`download` handles one file. `download_group` applies it to a named collection
and reports the expected total size. The `.part` suffix identifies an
incomplete transfer; only a validated file receives its final `.mat` name.

In [ ]:
def download(file_id: str, destination: Path, expected_size: int) -> None:
    """Stream one file and replace the destination only after validation."""
    if destination.exists() and destination.stat().st_size == expected_size:
        print(f"  skip  {destination.name} (already complete)")
        return

    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(f"{destination.name}.part")
    with requests.get(API.format(id=file_id), stream=True, timeout=60) as response:
        response.raise_for_status()
        received = 0
        with partial.open("wb") as stream, tqdm(
            total=expected_size,
            unit="B",
            unit_scale=True,
            desc=f"  {destination.name}",
            leave=True,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1 << 20):
                if not chunk:
                    continue
                stream.write(chunk)
                received += len(chunk)
                progress.update(len(chunk))

    if received != expected_size:
        raise OSError(
            f"Incomplete download for {destination.name}: received {received:,} "
            f"bytes, expected {expected_size:,}. Rerun this cell to try again."
        )
    partial.replace(destination)


def download_group(
    files: dict[str, tuple[str, int]], destination: Path, label: str
) -> None:
    """Download every file in a group, skipping complete local files."""
    total_gb = sum(size for _, size in files.values()) / 1e9
    print(f"{label} -> {destination}  (~{total_gb:.2f} GB)")
    for name, (file_id, size) in files.items():
        download(file_id, destination / name, expected_size=size)

## Download all recordings

Running this cell may take a long time depending on the connection. Progress
is shown for each file. If the kernel stops, rerun the cell: complete files are
skipped and incomplete files are downloaded again.

In [ ]:
download_group(CALCIUM_FILES, RAW_DIR, "Processed calcium recordings")
download_group(PHYSIOLOGY_FILES, PHYSIOLOGY_DIR, "Synchronized EEG/EMG recordings")

print("\nDone. Continue with 01_inspect_data.ipynb.")